In [0]:
%pip install optuna lightgbm xgboost catboost statsmodels scikit-learn mlflow
dbutils.library.restartPython()

In [0]:
# =========================================================================
# 1. IMPORTAÇÕES, PARÂMETROS E CONFIGURAÇÃO DO EXPERIMENT (SANDBOX)
# =========================================================================
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

import pandas as pd
import numpy as np

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from pyspark.sql import functions as F
from pyspark.sql.window import Window

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import optuna
import warnings
from datetime import date

# Coleta de parâmetros operacionais definidos no bundle
catalog_name = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema")

# Parametrização
PRODUCT_LIST = (
    "Colete Salva-Vidas 2374, Bateria Náutica 5523,Cabo Náutico 5921"
)
list_products = [p.strip() for p in PRODUCT_LIST.split(",")]

N_TRIALS = 25
TEST_MONTHS = 12
QUANTILE_ALPHA = 0.85
plot_product = "TODOS"

MAX_DATE = (date.today().replace(day=1) - pd.DateOffset(days=1)).date().isoformat()
# MAX_DATE = "2026-08-31"  # override manual

print(f"MAX_DATE em uso: {MAX_DATE}")

RANDOM_SEED = 42
MODEL_NAMES = ["LightGBM", "XGBoost", "CatBoost"]

# Define o registro de modelos como local/workspace (Sem Unity Catalog nesta fase)
# mlflow.set_registry_uri("databricks")
# mlflow.set_experiment("/Users/adrianostakata@gmail.com/sandbox_demand_store_product_forecasting")


# FUNÇÕES DE MÉTRICAS

def calculate_wape(y_true, y_pred):
    """Weighted Absolute Percentage Error"""
    total_actual = np.sum(np.abs(y_true))
    if total_actual == 0:
        return 0.0
    return np.sum(np.abs(y_true - y_pred)) / total_actual

def calculate_mape(y_true, y_pred):
    """Mean Absolute Percentage Error — calculado só sobre y_true > 0 para
    evitar divisão por zero (comum em demanda intermitente, com muitos meses
    de venda zero). Não é diretamente comparável ao WAPE: MAPE pesa cada
    observação igualmente, sem ponderar por volume — um mês de baixíssimo
    volume conta tanto quanto um mês de pico na média final."""
    y_true_arr = np.asarray(y_true, dtype=float)
    y_pred_arr = np.asarray(y_pred, dtype=float)
    mask = y_true_arr > 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true_arr[mask] - y_pred_arr[mask]) / y_true_arr[mask]))


# Caso a previsão seja negativa, substitua por zero
def safe_predict(model, X):
    """Previsões de demanda não podem ser negativas."""
    return np.maximum(model.predict(X), 0)

In [0]:
# LEITURA DOS DADOS AGRUPADOS POR ESTADO E PRODUTO
# Cada estado terá a série histórica completa de todos os produtos.
# As features temporais são calculadas por combinação estado + produto.

# Buscar vendas mensais agrupadas por estado e produto
sales_by_state_product = (
    spark.table("{catalog_name}.{schema}.fct_sales")
    .filter(F.col("is_effective_sale") == True)
    .filter(F.col("order_date") <= F.lit(MAX_DATE))
    .join(
        spark.table("{catalog_name}.{schema}d.dim_products"),
        "product_variant_id",
        "inner"
    )
    .join(
        spark.table("{catalog_name}.{schema}l.gold.dim_locations"),
        "location_id",
        "inner"
    )
    .withColumn("month_start", F.date_trunc("month", F.col("order_date")))
    .withColumn("month_num", F.month(F.col("month_start")))
    .withColumn("quarter_num", F.quarter(F.col("month_start")))
    .groupBy("month_start", "location_name", "product_name")
    .agg(
        F.sum("order_item_quantity").alias("quantity_sold"),
        F.first("month_num").alias("mes"),
        F.first("quarter_num").alias("trimestre")
    )
)

# Preencher meses faltantes com zeros após a primeira venda de cada produto por estado
# Determinar a primeira e última venda de cada combinação estado-produto
first_last_sales = (
    sales_by_state_product
    .groupBy("location_name", "product_name")
    .agg(
        F.min("month_start").alias("first_sale_month"),
        F.max("month_start").alias("last_sale_month")
    )
)

# Gerar todos os meses no intervalo completo dos dados
all_months = (
    sales_by_state_product
    .select(F.min("month_start").alias("min_month"))
    .collect()[0]
)

# Criar um DataFrame com todos os meses
months_list = spark.sql(f"""
    SELECT explode(sequence(
        date_trunc('month', date'{all_months.min_month}'),
        date_trunc('month', date'{MAX_DATE}'),
        interval 1 month
    )) as month_start
""").withColumn("month_num", F.month(F.col("month_start")))\
   .withColumn("quarter_num", F.quarter(F.col("month_start")))

# Cross join de estados-produtos únicos com todos os meses
state_product_combos = (
    sales_by_state_product
    .select("location_name", "product_name")
    .distinct()
)

all_state_product_months = (
    state_product_combos
    .crossJoin(months_list)
    .join(first_last_sales, ["location_name", "product_name"], "inner")
    # Filtrar apenas meses >= primeira venda e <= MAX_DATE
    .filter(F.col("month_start") >= F.col("first_sale_month"))
    .filter(F.col("month_start") <= F.lit(MAX_DATE))
    .select("location_name", "product_name", "month_start", "month_num", "quarter_num")
)

# Fazer left join para preencher meses sem vendas com 0
sales_by_state_product = (
    all_state_product_months
    .join(
        sales_by_state_product,
        ["location_name", "product_name", "month_start"],
        "left"
    )
    .withColumn("quantity_sold", F.coalesce(F.col("quantity_sold"), F.lit(0)))
    .withColumn("mes", F.coalesce(F.col("mes"), F.col("month_num")))
    .withColumn("trimestre", F.coalesce(F.col("trimestre"), F.col("quarter_num")))
    .select("month_start", "location_name", "product_name", "quantity_sold", "mes", "trimestre")
)

# Calcular features temporais usando Window functions
# Particiona por ESTADO e PRODUTO para que cada combinação tenha suas próprias features
window_spec = Window.partitionBy("location_name", "product_name").orderBy("month_start")
window_spec_rows_3 = window_spec.rowsBetween(-3, -1)
window_spec_rows_6 = window_spec.rowsBetween(-6, -1)
window_spec_rows_12 = window_spec.rowsBetween(-12, -1)

import math

featured_spark_df = (
    sales_by_state_product
    # Lags (valores históricos)
    .withColumn("lag_1", F.lag("quantity_sold", 1).over(window_spec))
    .withColumn("lag_12", F.lag("quantity_sold", 12).over(window_spec))
    # Rolling features (médias e estatísticas móveis)
    .withColumn("rolling_mean_3", F.avg("quantity_sold").over(window_spec_rows_3))  # reativada: usada como baseline de média móvel 3m
    .withColumn("rolling_mean_6", F.avg("quantity_sold").over(window_spec_rows_6))
    .withColumn("rolling_mean_12", F.avg("quantity_sold").over(window_spec_rows_12))
    .withColumn("rolling_std_12", F.stddev("quantity_sold").over(window_spec_rows_12))
    .withColumn("momentum", (F.col("lag_1") -  F.lag("quantity_sold", 2).over(window_spec)) / ( F.lag("quantity_sold", 2).over(window_spec) + F.lit(1.0)))
    # Codificação cíclica do mês — evita que dez (12) e jan (1) sejam vistos
    # como extremos opostos; complementa "mes"/"trimestre", não substitui.
    .withColumn("mes_sin", F.sin(F.col("mes") * (2 * math.pi / 12)))
    .withColumn("mes_cos", F.cos(F.col("mes") * (2 * math.pi / 12)))
    .dropna()
    .orderBy("location_name", "product_name", "month_start")
)

print(f"✓ Total de registros: {featured_spark_df.count():,}")
print(f"✓ Estados únicos: {featured_spark_df.select('location_name').distinct().count()}")
print(f"✓ Produtos únicos: {featured_spark_df.select('product_name').distinct().count()}")
print(f"✓ Combinações estado-produto: {featured_spark_df.select('location_name', 'product_name').distinct().count():,}")

display(featured_spark_df)

In [0]:
# PREPARAÇÃO PARA OS ALGORITMOS (PANDAS IN-MEMORY)
pdf = featured_spark_df.toPandas()

pdf["product_name"] = pdf["product_name"].astype("category")
pdf["location_name"] = pdf["location_name"].astype("category")

# Split Temporal — últimos N meses reservados para TESTE FINAL (holdout).
# Este conjunto só é usado uma vez, na avaliação final dos modelos otimizados
# nunca dentro do tuning do Optuna.
cutoff_date = pdf["month_start"].max() - pd.DateOffset(months=TEST_MONTHS)

train_df = pdf[pdf["month_start"] <= cutoff_date].reset_index(drop=True)
test_df = pdf[pdf["month_start"] > cutoff_date].reset_index(drop=True)

feature_cols = [
    "location_name", "product_name", "mes", "trimestre", "mes_sin", "mes_cos",
    "lag_1", "lag_12",
    "rolling_mean_6", "rolling_mean_12",
    "rolling_std_12",
    "momentum"]

target_col = "quantity_sold"

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

# Sem encoding: LightGBM, XGBoost e CatBoost lidam com product_name/location_name
# como categóricas nativamente (enable_categorical / cat_features).


In [0]:
# OTIMIZAÇÃO DE HIPERPARÂMETROS COM OPTUNA
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

VALIDATION_MONTHS = 12

train_df_sorted = pdf.sort_values("month_start").reset_index(drop=True)
validation_cutoff = train_df_sorted["month_start"].max() - pd.DateOffset(months=VALIDATION_MONTHS)

fit_df = train_df_sorted[train_df_sorted["month_start"] <= validation_cutoff]
val_df = train_df_sorted[train_df_sorted["month_start"] > validation_cutoff]

X_fit, y_fit = fit_df[feature_cols], fit_df[target_col]
X_val, y_val = val_df[feature_cols], val_df[target_col]

print(f"Ajuste (fit): {len(X_fit)} linhas, até {fit_df['month_start'].max().strftime('%Y-%m')}")
print(f"Validação: {len(X_val)} linhas, de {val_df['month_start'].min().strftime('%Y-%m')} "
      f"a {val_df['month_start'].max().strftime('%Y-%m')}\n")

def simple_wape(clf_builder, X_fit, y_fit, X_val, y_val):
    """WAPE num único split treino/validação (sem cross-validation)."""
    clf = clf_builder()
    clf.fit(X_fit, y_fit)
    preds = np.maximum(clf.predict(X_val), 0)
    return calculate_wape(y_val, preds)

def create_objective(model_name):
    """Cria função objetivo para Optuna baseada no modelo selecionado."""
    # Treina para o quantil QUANTILE_ALPHA (não a média condicional)
    # em vez de RMSE/L2, que "achata" a previsão perto da média
    # e ignora os picos raros de demanda.
    def objective(trial):
        if model_name == "LightGBM":
            params = {
                "objective": "quantile",
                "alpha": QUANTILE_ALPHA,
                "n_estimators": trial.suggest_int("n_estimators", 300, 800, step=50),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 15, 127),
                "max_depth": trial.suggest_int("max_depth", 3, 15),
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
                "random_state": RANDOM_SEED,
                "verbose": -1
            }
            builder = lambda: lgb.LGBMRegressor(**params)
            return simple_wape(builder, X_fit, y_fit, X_val, y_val)

        elif model_name == "XGBoost":
            params = {
                "objective": "reg:quantileerror",
                "quantile_alpha": QUANTILE_ALPHA,
                "tree_method": "hist",  # exigido por reg:quantileerror
                "n_estimators": trial.suggest_int("n_estimators", 500, 1000, step=50),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "max_depth": trial.suggest_int("max_depth", 3, 15),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
                "enable_categorical": True,
                "random_state": RANDOM_SEED
            }
            builder = lambda: xgb.XGBRegressor(**params)
            return simple_wape(builder, X_fit, y_fit, X_val, y_val)

        elif model_name == "CatBoost":
            params = {
                "loss_function": f"Quantile:alpha={QUANTILE_ALPHA}",
                "iterations": trial.suggest_int("iterations", 400, 800, step=50),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "depth": trial.suggest_int("depth", 3, 10),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
                "cat_features": ["product_name", "location_name"],
                "verbose": 0,
                "random_seed": RANDOM_SEED
            }
            builder = lambda: CatBoostRegressor(**params)
            return simple_wape(builder, X_fit, y_fit, X_val, y_val)

    return objective

# Executar otimização para cada modelo
best_params = {}

print(f"Treinando para o quantil alpha={QUANTILE_ALPHA} (LightGBM/XGBoost/CatBoost).\n")

with mlflow.start_run(run_name="optuna_hpo_parent"):

    for model_name in MODEL_NAMES:
        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
            study_name=f"optuna_{model_name}"
        )
        study.optimize(create_objective(model_name), n_trials=N_TRIALS)

        best_params[model_name] = study.best_params.copy()
        best_params[model_name]["random_state"] = RANDOM_SEED

        with mlflow.start_run(run_name=f"HPO_{model_name}", nested=True):
            mlflow.log_params({f"{model_name}_{k}": v for k, v in study.best_params.items()})
            mlflow.log_param("quantile_alpha", QUANTILE_ALPHA)
            mlflow.log_metric("best_validation_wape_no_cv", study.best_value)

        print(f"\n{'='*60}")
        print(f"Modelo: {model_name}")
        print(f"Melhor WAPE (validação simples, sem CV): {study.best_value:.4f}")
        print(f"Melhores Parâmetros: {study.best_params}")

In [0]:
# TREINAR MODELOS OTIMIZADOS (TREINO COMPLETO) E AVALIAR NO HOLDOUT
optimized_models = {}
optuna_results = []

for model_name in MODEL_NAMES:
    params = best_params[model_name].copy()

    if model_name == "LightGBM":
        params["objective"] = "quantile"
        params["alpha"] = QUANTILE_ALPHA
        params["verbose"] = -1
        optimized_models[model_name] = {"model": lgb.LGBMRegressor(**params)}
    elif model_name == "XGBoost":
        params["objective"] = "reg:quantileerror"
        params["quantile_alpha"] = QUANTILE_ALPHA
        params["tree_method"] = "hist"
        params["enable_categorical"] = True
        optimized_models[model_name] = {"model": xgb.XGBRegressor(**params)}
    elif model_name == "CatBoost":
        params["loss_function"] = f"Quantile:alpha={QUANTILE_ALPHA}"
        params["cat_features"] = ["product_name", "location_name"]
        params["verbose"] = 0
        optimized_models[model_name] = {"model": CatBoostRegressor(**params)}

with mlflow.start_run(run_name="optuna_eval_parent"):

    # Baseline ingênuo — calculado uma vez, fora do loop (não muda por modelo).
    # Serve de referência: se um modelo não bate isso por margem boa, ele não
    # está agregando valor real, independente do WAPE absoluto dele. Também
    # ajuda a interpretar o gap treino→teste: um gap alto num período de teste
    # que também é difícil para o baseline não é necessariamente overfitting.
    naive_wape_test = calculate_wape(y_test, X_test["lag_1"])
    mlflow.log_metric("naive_baseline_wape", naive_wape_test)
    print(f"Baseline ingênuo (repetir lag_1) no teste: {naive_wape_test*100:.2f}%\n")

    for model_name, config in optimized_models.items():
        with mlflow.start_run(run_name=f"Optimized_{model_name}", nested=True):

            clf = config["model"]

            clf.fit(X_train, y_train)
            preds = safe_predict(clf, X_test)  # único toque no holdout: avaliação final

            wape = calculate_wape(y_test, preds)
            mape = calculate_mape(y_test, preds)
            rmse = np.sqrt(mean_squared_error(y_test, preds))
            mae = mean_absolute_error(y_test, preds)

            # DIAGNÓSTICO DE OVERFITTING: erro no TREINO vs. erro no TESTE
            preds_train = safe_predict(clf, X_train)
            wape_train = calculate_wape(y_train, preds_train)

            gap_pp = (wape - wape_train) * 100  # gap em pontos percentuais
            gap_relativo = (wape / wape_train - 1) * 100 if wape_train > 0 else np.nan
            supera_baseline_pp = (naive_wape_test - wape) * 100  # positivo = modelo melhor que o baseline

            mlflow.log_metrics({
                "train_wape": wape_train,
                "test_wape": wape,
                "overfit_gap_pp": gap_pp,
                "test_mape": mape,
                "test_rmse": rmse,
                "test_mae": mae
            })

            signature = infer_signature(X_test, preds)
            mlflow.sklearn.log_model(clf, artifact_path="model", signature=signature, serialization_format="cloudpickle")

            optuna_results.append({
                "Modelo": model_name,
                "WAPE Treino (%)": round(wape_train * 100, 2),
                "WAPE Teste (%)": round(wape * 100, 2),
                "WAPE (%)": round(wape * 100, 2),  # mantido p/ compatibilidade com as demais células
                "Baseline Ingênuo (%)": round(naive_wape_test * 100, 2),
                "Supera Baseline (p.p.)": round(supera_baseline_pp, 2),
                "Gap Treino-Teste (p.p.)": round(gap_pp, 2),
                "MAPE (%)": round(mape * 100, 2),
                "MAE": round(mae, 4),
                "RMSE": round(rmse, 4)
            })

            # Veredicto por modelo — cruza o gap treino→teste com o quanto o
            # modelo bate o baseline ingênuo. Um gap grande só é motivo de
            # alarme se o modelo também não estiver superando o baseline por
            # margem boa (senão é o teste que é difícil, não o modelo que
            # está memorizando).
            if wape_train == 0 and wape > 0:
                veredicto = "WAPE de treino ZERO — memorização quase certa (verificar profundidade/regularização)."
            elif gap_relativo >= 100 and supera_baseline_pp < 5:
                veredicto = (f"Erro no teste é {gap_relativo:.0f}% maior que no treino E mal supera o "
                             f"baseline ingênuo (+{supera_baseline_pp:.1f} p.p.) — indício forte de overfitting.")
            elif gap_relativo >= 100:
                veredicto = (f"Gap treino→teste alto ({gap_relativo:.0f}%), mas o modelo supera o baseline "
                             f"ingênuo por {supera_baseline_pp:.1f} p.p. — provavelmente reflete dificuldade "
                             f"do período de teste, não memorização.")
            elif gap_relativo >= 40:
                veredicto = f"Erro no teste {gap_relativo:.0f}% maior que no treino — vale atenção, mas não é grave."
            else:
                veredicto = "Gap treino→teste dentro do esperado — sem sinal forte de overfitting."

            print(f"{model_name}: WAPE treino={wape_train*100:.2f}% | WAPE teste={wape*100:.2f}% | "
                  f"baseline ingênuo={naive_wape_test*100:.2f}% | {veredicto}")

# TABELA COMPARATIVA DOS MODELOS OTIMIZADOS (HOLDOUT)
optuna_results_df = pd.DataFrame(optuna_results).sort_values(by="WAPE (%)").reset_index(drop=True)
print(f"Modelos treinados com objective=quantile alpha={QUANTILE_ALPHA} — a previsão "
      f"mira deliberadamente acima da mediana (margem de segurança para estoque).")
display(optuna_results_df)

# BASELINES INGÊNUOS — MESMAS MÉTRICAS DOS MODELOS, PARA COMPARAÇÃO DIRETA

baselines = {
    "Baseline: repetir mês anterior (lag_1)": "lag_1",
}

baseline_results = []
for nome, coluna in baselines.items():
    pred_treino = X_train[coluna].values
    pred_teste = X_test[coluna].values

    wape_b_treino = calculate_wape(y_train, pred_treino)
    wape_b_teste = calculate_wape(y_test, pred_teste)
    mape_b_teste = calculate_mape(y_test, pred_teste)
    mae_b_teste = mean_absolute_error(y_test, pred_teste)
    rmse_b_teste = np.sqrt(mean_squared_error(y_test, pred_teste))

    baseline_results.append({
        "Modelo": nome,
        "WAPE Treino (%)": round(wape_b_treino * 100, 2),
        "WAPE Teste (%)": round(wape_b_teste * 100, 2),
        "WAPE (%)": round(wape_b_teste * 100, 2),
        "MAPE (%)": round(mape_b_teste * 100, 2),
        "MAE": round(mae_b_teste, 4),
        "RMSE": round(rmse_b_teste, 4),
    })

baseline_results_df = pd.DataFrame(baseline_results)

print("\nBaselines ingênuos (mesmas métricas, calculadas sobre o holdout):")
display(baseline_results_df)

# Comparativo completo — modelos de ML + baselines juntos, ordenado por WAPE
# de teste. Colunas exclusivas dos modelos (Gap Treino-Teste, Supera Baseline)
# ficam em branco (NaN) nas linhas de baseline — não se aplicam a eles.
comparativo_completo_df = (
    pd.concat([optuna_results_df, baseline_results_df], ignore_index=True)
    .sort_values("WAPE (%)")
    .reset_index(drop=True)
)
print("\nComparativo completo — ML + baselines:")
display(comparativo_completo_df)

In [0]:
# GRÁFICO COMPARATIVO: REAL vs. PREDIÇÕES POR ESTABELECIMENTO E PRODUTO
# Filtrar apenas os produtos de interesse (list_products definida em outra célula)
pdf_filtrado = pdf[pdf["product_name"].isin(list_products)].copy()

X_full = pdf_filtrado[feature_cols].copy()

predictions = {"data": pdf_filtrado["month_start"].values, "Real": pdf_filtrado["quantity_sold"].values}

for model_name, config in optimized_models.items():
    clf = config["model"]
    predictions[model_name] = safe_predict(clf, X_full)

pred_df = pd.DataFrame(predictions)
pred_df["product_name"] = pdf_filtrado["product_name"].values
pred_df["location_name"] = pdf_filtrado["location_name"].values

# Usar TODOS os produtos de TODOS os estabelecimentos
pred_df = pred_df.copy()

model_names = list(optimized_models.keys())
estabelecimentos = sorted(pred_df["location_name"].unique())

# Paleta de cores — uma cor por série (Real + cada modelo)
series_labels = ["Real"] + model_names
series_colors = plt.cm.tab10(np.linspace(0, 1, len(series_labels)))
color_map = dict(zip(series_labels, series_colors))

# Estilos de linha por série
line_styles = {"Real": "-"}
for m in model_names:
    line_styles[m] = "--"

n_total = sum(
    1 for estab in estabelecimentos
    for product in pred_df[pred_df["location_name"] == estab]["product_name"].unique()
)

print(f"Gerando {n_total} gráfico(s) — um por estabelecimento × produto, "
      f"comparando Real vs. {', '.join(model_names)}.\n")

for estab in estabelecimentos:
    estab_data = pred_df[pred_df["location_name"] == estab].copy()
    products_in_estab = sorted(estab_data["product_name"].unique())

    for product in products_in_estab:
        product_data = estab_data[estab_data["product_name"] == product].copy()
        product_data = product_data.sort_values("data")

        plt.figure(figsize=(16, 6))

        # Linha sólida = série real; linhas tracejadas = predições de cada modelo
        for label in series_labels:
            plt.plot(product_data["data"], product_data[label],
                     label=label, linewidth=2 if label == "Real" else 1.5,
                     color=color_map[label], linestyle=line_styles[label],
                     marker="o" if label == "Real" else "", markersize=2,
                     alpha=1.0 if label == "Real" else 0.7)

        plt.axvline(x=cutoff_date, color="gray", linestyle=":", linewidth=1.5,
                    label=f"Início do Teste ({TEST_MONTHS} meses)")

        plt.title(f"Real vs. Modelos — {product} | {estab}", fontsize=14, fontweight="bold")
        plt.xlabel("Data", fontsize=12)
        plt.ylabel("Quantidade Vendida", fontsize=12)
        plt.legend(loc="best", fontsize=9)
        plt.grid(True, alpha=0.3)
        # Garantir que todos os meses sejam exibidos, incluindo agosto
        ax = plt.gca()
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))  # a cada 2 meses para não poluir
        # Ajustar os limites do eixo X para garantir que agosto apareça
        ax.set_xlim(product_data["data"].min(), product_data["data"].max())
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

In [0]:
# EMPACOTAR O CAMPEÃO NUM PYFUNC E REGISTRAR NO UNITY CATALOG
class DemandForecastModel(mlflow.pyfunc.PythonModel):
    """Wrapper de serving do forecast de demanda."""

    def __init__(self, model, feature_cols, product_categories, location_categories, model_name, quantile_alpha):
        self.model = model
        self.feature_cols = list(feature_cols)
        self.product_categories = list(product_categories)
        self.location_categories = list(location_categories)
        self.model_name = model_name
        self.quantile_alpha = quantile_alpha

    def _prepare(self, df):
        faltando = [c for c in self.feature_cols if c not in df.columns]
        if faltando:
            raise ValueError(f"Colunas ausentes na entrada: {faltando}")

        X = df[self.feature_cols].copy()
        X["product_name"] = X["product_name"].astype(str)
        X["location_name"] = X["location_name"].astype(str)

        # CatBoost espera string em cat_features; LGBM/XGB esperam a MESMA
        # categoria (mesma ordem) usada no fit — daí guardarmos as categorias.
        if self.model_name != "CatBoost":
            X["product_name"] = pd.Categorical(
                X["product_name"], categories=self.product_categories
            )
            X["location_name"] = pd.Categorical(
                X["location_name"], categories=self.location_categories
            )

        for c in self.feature_cols:
            if c not in ("product_name", "location_name"):
                X[c] = pd.to_numeric(X[c], errors="coerce")
        return X

    def predict(self, context, model_input, params=None):
        if not isinstance(model_input, pd.DataFrame):
            model_input = pd.DataFrame(model_input)
        X = self._prepare(model_input)
        return np.maximum(self.model.predict(X), 0)


product_categories = sorted(pdf["product_name"].astype(str).unique().tolist())
best_model_name = optuna_results_df.iloc[0]["Modelo"]
location_categories = sorted(pdf["location_name"].astype(str).unique().tolist())



pyfunc_model = DemandForecastModel(
    model=optimized_models[best_model_name]["model"],
    feature_cols=feature_cols,
    product_categories=product_categories,
    location_categories=location_categories,
    model_name=best_model_name,
    quantile_alpha=QUANTILE_ALPHA,
)

# Teste rápido: o wrapper tem que reproduzir o mesmo número do notebook.
amostra = test_df[feature_cols].head(5).copy()
amostra["product_name"] = amostra["product_name"].astype(str)

pred_wrapper = pyfunc_model.predict(None, amostra)
pred_direto = safe_predict(optimized_models[best_model_name]["model"], X_test.head(5))

print(f"Campeão: {best_model_name}")
print(f"Wrapper : {np.round(pred_wrapper, 4)}")
print(f"Direto  : {np.round(pred_direto, 4)}")
assert np.allclose(pred_wrapper, pred_direto), (
    "Wrapper divergiu do predict original — investigar o tratamento de product_name."
)
print("\n✓ Wrapper consistente com o pipeline do notebook.")

# -------------------------------------------------------------------------
# Registrar no Unity Catalog e apontar o alias @champion
# -------------------------------------------------------------------------
from mlflow.tracking import MlflowClient
import sklearn, cloudpickle, lightgbm, xgboost, catboost

# O registry do workspace está desabilitado — usamos o UC.
mlflow.set_registry_uri("databricks-uc")

MODEL_NAME = "demand_forecast_store_product"
UC_MODEL = f"{catalog_name}.{schema}.{MODEL_NAME}"

# input_example define o schema que o endpoint vai validar nas requisições.
input_example = test_df[feature_cols].head(3).copy()
input_example["product_name"] = input_example["product_name"].astype(str)
input_example["location_name"] = input_example["location_name"].astype(str)
signature = infer_signature(input_example, pyfunc_model.predict(None, input_example))

# Dependências explícitas: o pyfunc não "enxerga" lightgbm/xgboost/catboost
# sozinho, e endpoint que sobe sem elas falha na construção do container.
pip_requirements = [
    f"mlflow=={mlflow.__version__}",
    f"pandas=={pd.__version__}",
    f"numpy=={np.__version__}",
    f"scikit-learn=={sklearn.__version__}",
    f"cloudpickle=={cloudpickle.__version__}",
    f"lightgbm=={lightgbm.__version__}",
    f"xgboost=={xgboost.__version__}",
    f"catboost=={catboost.__version__}",
]

with mlflow.start_run(run_name="register_serving_model") as run:
    mlflow.log_params({
        "champion_model": best_model_name,
        "quantile_alpha": QUANTILE_ALPHA,
        "test_months": TEST_MONTHS,
        "max_date": MAX_DATE,
        "n_features": len(feature_cols),
        "n_products": len(product_categories),
    })
    mlflow.log_metric(
        "holdout_wape_pct",
        float(optuna_results_df.loc[
            optuna_results_df["Modelo"] == best_model_name, "WAPE (%)"
        ].values[0]),
    )

    model_info = mlflow.pyfunc.log_model(
        artifact_path="serving_model",
        python_model=pyfunc_model,
        signature=signature,
        input_example=input_example,
        pip_requirements=pip_requirements,
        registered_model_name=UC_MODEL,
    )

version = model_info.registered_model_version
client = MlflowClient()

# Alias em vez de número fixo: trocar de versão depois não exige mexer no endpoint.
client.set_registered_model_alias(UC_MODEL, "champion", version)
client.update_model_version(
    name=UC_MODEL,
    version=version,
    description=(
        f"Forecast de demanda mensal por produto e estabelecimento. Algoritmo: {best_model_name}, "
        f"treinado no quantil alpha={QUANTILE_ALPHA} (nível de serviço). "
        f"Saída já clipada em 0."
    ),
)

print(f"Registrado: {UC_MODEL} versão {version}")
print(f"Alias @champion → versão {version}")
print(f"URI para consumo: models:/{UC_MODEL}@champion")

In [0]:
# FUNÇÃO DE PREVISÃO RECURSIVA MULTI-STEP
import math
import numpy as np
import pandas as pd

HORIZON_MONTHS = 3  # quantos meses prever à frente
LIMIAR_ZERO = 0.5  # previsão contínua abaixo disso vira 0 em vez de ceil

def _trimestre(mes: int) -> int:
    return (mes - 1) // 3 + 1

def prever_proximos_meses(pdf_historico, modelo_wrapper, feature_cols, horizon=3, min_history=12):
    """Previsão recursiva multi-step, produto a produto.

    A cada passo do horizonte:
      1. monta as features do próximo mês a partir do histórico
         (vendas reais + previsões já geradas nesta mesma chamada);
      2. chama o modelo campeão via wrapper (já aplica o clip em 0);
      3. anexa a previsão ao histórico, para servir de lag/rolling do
         próximo passo.

    Produtos com menos de `min_history` meses ficam de fora — lag_12 não
    pode ser calculado com menos de 12 meses de histórico.
    """
    resultados = []

    for (produto, loja), hist in pdf_historico.sort_values("month_start").groupby(["product_name", "location_name"]):
        serie = hist["quantity_sold"].tolist()

        if len(serie) < min_history:
            #print(f"{produto} ({loja}): {len(serie)} meses de histórico (< {min_history}) — pulando.")
            continue

        ultimo_mes = hist["month_start"].max()

        for h in range(1, horizon + 1):
            mes_alvo = ultimo_mes + pd.DateOffset(months=h)

            janela_3 = serie[-3:]
            janela_6 = serie[-6:]
            janela_12 = serie[-12:]

            rolling_mean_3 = float(np.mean(janela_3))
            rolling_mean_6 = float(np.mean(janela_6))
            rolling_mean_12 = float(np.mean(janela_12))

            linha = pd.DataFrame([{
                "product_name": produto,
                "location_name": loja,
                "mes": mes_alvo.month,
                "trimestre": _trimestre(mes_alvo.month),
                "lag_1": serie[-1],
                "lag_2": serie[-2],
                "lag_3": serie[-3],
                "lag_12": serie[-12],
                "rolling_mean_3": rolling_mean_3,
                "rolling_mean_6": rolling_mean_6,
                "rolling_mean_12": rolling_mean_12,
                "rolling_std_6": float(np.std(janela_6, ddof=0)),
                "rolling_std_12": float(np.std(janela_12, ddof=0)),
                "rolling_max_6": float(np.max(janela_6)),
                "rolling_max_12": float(np.max(janela_12)),
                "momentum": (serie[-1] - serie[-2]) / (serie[-2] + 1.0),
                "mes_sin": math.sin(mes_alvo.month * (2 * math.pi / 12)),
                "mes_cos": math.cos(mes_alvo.month * (2 * math.pi / 12)),
            }])[feature_cols]

            pred_continua = float(modelo_wrapper.predict(None, linha)[0])

            if pred_continua < LIMIAR_ZERO:
                pred_inteira = 0
            else:
                pred_inteira = math.ceil(pred_continua)

            resultados.append({
                "product_name": produto,
                "location_name": loja,
                "month_start": mes_alvo.normalize(),
                "demand_forcast": round(pred_continua, 4),  # mais casas p/ auditar o limiar
                "unit_demand_forcast": pred_inteira,
            })

            serie.append(pred_inteira)

    return pd.DataFrame(resultados)

# RODAR A PREVISÃO PARA O HORIZONTE CONFIGURADO
forecast_rolling_df = prever_proximos_meses(
    pdf_historico=pdf,
    modelo_wrapper=pyfunc_model,
    feature_cols=feature_cols,
    horizon=HORIZON_MONTHS,
)

ultimo_mes_fechado = pdf["month_start"].max()
print(f"Último mês fechado nos dados de treino: {ultimo_mes_fechado.strftime('%Y-%m')}")
print(f"Previsão gerada para os próximos {HORIZON_MONTHS} meses "
      f"({(ultimo_mes_fechado + pd.DateOffset(months=1)).strftime('%Y-%m')} a "
      f"{(ultimo_mes_fechado + pd.DateOffset(months=HORIZON_MONTHS)).strftime('%Y-%m')})\n")

display(forecast_rolling_df)

In [0]:
# PERSISTIR EM DELTA (UPSERT POR product_name + month_start)
from pyspark.sql import functions as F
from delta.tables import DeltaTable

TABELA_FORECAST_ROLLING = f"{catalog_name}.{schema}.demand_forecast_store_product_rolling_3m"

forecast_spark_df = (
    spark.createDataFrame(forecast_rolling_df)
    .withColumn("model_name", F.lit(best_model_name))
    .withColumn("model_version", F.lit(str(version)))
    .withColumn("generated_at", F.current_timestamp())
)

display(forecast_spark_df)

if spark.catalog.tableExists(TABELA_FORECAST_ROLLING):
    existing_cols = [f.name for f in spark.table(TABELA_FORECAST_ROLLING).schema.fields]
    if "location_name" not in existing_cols:
        spark.sql(f"DROP TABLE IF EXISTS {TABELA_FORECAST_ROLLING}")
        forecast_spark_df.write.saveAsTable(TABELA_FORECAST_ROLLING)
        print(f"✓ Tabela recriada com location_name: {TABELA_FORECAST_ROLLING}")
    else:
        delta_tbl = DeltaTable.forName(spark, TABELA_FORECAST_ROLLING)
        (
            delta_tbl.alias("destino")
            .merge(
                forecast_spark_df.alias("novo"),
                "destino.product_name = novo.product_name AND destino.location_name = novo.location_name AND destino.month_start = novo.month_start"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"✓ Upsert em {TABELA_FORECAST_ROLLING}")
else:
    forecast_spark_df.write.saveAsTable(TABELA_FORECAST_ROLLING)
    print(f"✓ Tabela criada: {TABELA_FORECAST_ROLLING}")

display(
    spark.table(TABELA_FORECAST_ROLLING)
    .orderBy("product_name", "month_start")
)

In [0]:
# CONCATENAR HISTÓRICO COMPLETO + PREVISÃO DE 3 MESES
# Monta um dataframe único com toda a série histórica de vendas por
# produto/estabelecimento e, no final de cada série, as 3 previsões
# geradas pelo modelo campeão.

# --- Colunas padronizadas do histórico ---
hist_cols = ["product_name", "location_name", "month_start", "quantity_sold"]
hist_df = pdf[hist_cols].copy()
hist_df["is_real"] = True

# --- Colunas padronizadas da previsão ---
fcst_cols = ["product_name", "location_name", "month_start", "unit_demand_forcast"]
fcst_df = forecast_rolling_df[fcst_cols].copy()
fcst_df["quantity_sold"] = fcst_df["unit_demand_forcast"]  # valor previsto (inteiro) para alinhar com quantity_sold
fcst_df["is_real"] = False

# --- Empilhar histórico + previsão ---
colunas_finais = [
    "product_name", "location_name", "month_start",
    "is_real", "quantity_sold"
]
serie_completa = pd.concat(
    [hist_df[colunas_finais], fcst_df[colunas_finais]],
    ignore_index=True
).sort_values(["product_name", "location_name", "month_start"]).reset_index(drop=True)

print(f"Total de linhas: {len(serie_completa)}")
print(f"Linhas reais:    {len(hist_df)}")
print(f"Linhas previstas: {len(fcst_df)}")
print()
display(serie_completa)

# =========================================================================
# PERSISTIR EM DELTA (dataset_demand_full)
# =========================================================================
TABELA_SERIE_COMPLETA = f"{catalog_name}.{schema}.dataset_demand_store_product_full"

serie_completa_spark = spark.createDataFrame(serie_completa)

if spark.catalog.tableExists(TABELA_SERIE_COMPLETA):
    existing_cols = [f.name for f in spark.table(TABELA_SERIE_COMPLETA).schema.fields]
    expected_cols = ["product_name", "location_name", "month_start", "is_real", "quantity_sold"]
    if existing_cols != expected_cols:
        spark.sql(f"DROP TABLE IF EXISTS {TABELA_SERIE_COMPLETA}")
        serie_completa_spark.write.saveAsTable(TABELA_SERIE_COMPLETA)
        print(f"✓ Tabela recriada: {TABELA_SERIE_COMPLETA}")
    else:
        delta_tbl = DeltaTable.forName(spark, TABELA_SERIE_COMPLETA)
        (
            delta_tbl.alias("destino")
            .merge(
                serie_completa_spark.alias("novo"),
                "destino.product_name = novo.product_name AND destino.location_name = novo.location_name AND destino.month_start = novo.month_start"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"✓ Upsert em {TABELA_SERIE_COMPLETA}")
else:
    serie_completa_spark.write.saveAsTable(TABELA_SERIE_COMPLETA)
    print(f"Tabela criada: {TABELA_SERIE_COMPLETA}")

display(
    spark.table(TABELA_SERIE_COMPLETA)
    .orderBy("product_name", "location_name", "month_start")
)